# 개인실습 — 한빛은행 창구 운영 데이터 · 정답 모음

> 이 노트북은 `NoSQL_Redis_5_한빛은행_창구_운영_데이터_개인실습.ipynb`(문제 노트북)의 **정답**입니다. `db-redis` 컨테이너가 실행 중이어야 합니다(`NoSQL_Redis_4_기본_자료구조.ipynb`의 연결 셀과 같은 방식으로 접속합니다).


In [ ]:
# ✅ 연결 준비 — NoSQL_Redis_4_기본_자료구조.ipynb와 동일
import redis, time

r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
print("PING ->", r.ping())


## 실습 1 정답 — 일회용 인증번호 (String + TTL)

In [ ]:
# 실습 1: 일회용 인증번호 — String에 TTL(수명)을 걸어 "한동안만 유효한 값"을 구현합니다.
# ex=180: 180초(3분) 뒤 Redis가 이 키를 자동으로 지웁니다 — 문자메시지로 받는 인증번호가
# 시간이 지나면 쓸 수 없게 되는 것과 같은 동작을, 사람이 직접 지우지 않아도 Redis가 대신 처리해줍니다.
r.set("auth:cust:2001", "483920", ex=180)
print("남은 수명 ->", r.ttl("auth:cust:2001"))   # → 180 (또는 그 이하)
print("인증번호 ->", r.get("auth:cust:2001"))     # → 483920
# 💡 3분 뒤 다시 r.get(...) 하면 None — 인증번호가 저절로 무효화됩니다


## 실습 2 정답 — 창구 상담 카드 (Hash)

In [ ]:
# 실습 2: 창구 상담 카드 — Hash로 고객 한 명의 여러 정보(이름/목적/방문횟수)를 한 key에 묶어 관리합니다.
# mapping={...}으로 필드 3개(name/purpose/visits)를 한 번에 저장 — 4교시 Hash 예제와 같은 방식입니다.
r.hset("counter:cust:2001", mapping={"name": "이서준", "purpose": "계좌개설", "visits": "1"})
# hincrby(key, 필드, 증가량): visits 필드를 정수로 취급해 1 증가 — "재방문"을 기록하는 셈입니다.
r.hincrby("counter:cust:2001", "visits", 1)
print("상담 카드 ->", r.hgetall("counter:cust:2001"))       # hgetall: 이 Hash의 모든 필드를 한 번에 조회
print("목적만 ->", r.hget("counter:cust:2001", "purpose"))  # hget: 필드 하나만 콕 집어 조회
# → 상담 카드 -> {'name': '이서준', 'purpose': '계좌개설', 'visits': '2'} / 목적만 -> 계좌개설


## 실습 3 정답 — 대기 번호표 (List, FIFO)

In [ ]:
# 실습 3: 대기 번호표 — List로 "번호표 발급 순서 = 창구에서 처리할 순서"를 구현합니다(FIFO).
# lpush로 왼쪽(맨 앞)에 계속 밀어넣으면, 가장 나중에 lpush한 값(103)이 맨 왼쪽에 위치하게 됩니다.
r.lpush("waiting:branch1", "101")   # 첫 번째로 발급된 번호표
r.lpush("waiting:branch1", "102")
r.lpush("waiting:branch1", "103")   # 가장 최근에 발급된 번호표 — 맨 왼쪽(인덱스 0)에 위치
print("대기 인원 ->", r.llen("waiting:branch1"))                 # → 3
# brpop: 오른쪽(맨 뒤)에서 하나 꺼내며 제거 — 맨 왼쪽에 쌓고 맨 오른쪽에서 꺼내므로, 가장 먼저 발급된
# 번호(101)가 가장 먼저 나옵니다 — "먼저 뽑은 번호표가 먼저 불린다"는 실제 은행 창구 운영과 정확히 일치합니다.
print("다음 호출 ->", r.brpop("waiting:branch1", timeout=1))      # → ('waiting:branch1', '101') — 101이 먼저!
print("남은 대기열 ->", r.lrange("waiting:branch1", 0, -1))       # → ['103', '102']


## 실제로 채점해보기 — 실습 1~3을 이어서 실행

In [ ]:
# ✅ 위 세 실습을 정답으로 순서대로 실행해 실제 동작을 확인합니다.
# (이 셀 하나로 실습 1~3을 처음부터 다시 실행 — 위에서 이미 실행했어도 키를 다시 만들므로 안전합니다)
r.set("auth:cust:2001", "483920", ex=180)
print("[실습1] 남은 수명 ->", r.ttl("auth:cust:2001"), "| 인증번호 ->", r.get("auth:cust:2001"))

r.delete("counter:cust:2001")   # 이전 실행에서 visits가 이미 2 이상으로 늘어나 있을 수 있으므로 초기화 후 재시작
r.hset("counter:cust:2001", mapping={"name": "이서준", "purpose": "계좌개설", "visits": "1"})
r.hincrby("counter:cust:2001", "visits", 1)
print("[실습2] 상담 카드 ->", r.hgetall("counter:cust:2001"))

r.delete("waiting:branch1")   # 이전 대기열이 남아있으면 순서 확인이 헷갈리므로 비우고 다시 시작
r.lpush("waiting:branch1", "101")
r.lpush("waiting:branch1", "102")
r.lpush("waiting:branch1", "103")
print("[실습3] 대기 인원 ->", r.llen("waiting:branch1"))
print("[실습3] 다음 호출 ->", r.brpop("waiting:branch1", timeout=1), "(101이 먼저 나와야 정상)")
print("[실습3] 남은 대기열 ->", r.lrange("waiting:branch1", 0, -1))


## 마무리

- **String + TTL**: 인증번호처럼 "잠깐 쓰고 저절로 사라져야 하는 값"에 맞습니다.
- **Hash**: 상담 카드처럼 "한 대상에 여러 속성을 묶어서" 다룰 때 맞습니다.
- **List(LPUSH+BRPOP)**: 번호표처럼 "발급 순서 = 처리 순서"가 중요할 때 맞습니다.

세 자료구조 모두 `NoSQL_Redis_4_기본_자료구조.ipynb`에서 처음 만난 것과 완전히 같은 명령입니다 — 데이터 성격만 실제 업무(창구 운영)로 바뀌었습니다.
